# DeepTrees Baseline — Zero-Shot auf Kiel-Daten

Dieses Notebook führt das vortrainierte **freudenberg2022**-Modell (U-Net ResNet18, 4-Kanal RGBI, trainiert auf 20cm-Daten aus Halle) zero-shot auf den Kiel-Test-Gebieten aus und misst die Erkennungsleistung auf Instanz-Ebene.

**Test-Gebiete:** BotGarten, HoernNord  (`TrainingAreas/test/`)

**Pipeline:**
1. Modell herunterladen und laden
2. Kachel einlesen (RGBI GeoTIFF, 7.5 cm oder 20 cm) → ggf. auf 20 cm herunterskalieren
3. Inferenz: Maske + Outline + Distanztransformation
4. Post-Processing: Watershed → Baumkronen-Polygone
5. Evaluation: IoU-Matching gegen GT → TP / FP / FN / Precision / Recall / F1

## 0 — Konfiguration

Hier alle Pfade und Parameter eintragen. **Nur diese Zelle muss angepasst werden.**

In [1]:
from pathlib import Path

# ── Basis-Pfade ────────────────────────────────────────────────────────────
BASE = Path("../Data/Kiel/TrainingAreas")

# Test-Gebiete
TEST_AREAS = ["BotGarten", "HoernNord"]

# Auflösung: "7.5cm" oder "20cm"
RESOLUTION = "7.5cm"

TILES_DIR = BASE / ("DOP7-5" if RESOLUTION == "7.5cm" else "DOP20")
GT_DIR    = BASE / "test"

# Modellgewichte — in ndas Home, da das Notebook als nda läuft
MODEL_PATH = Path.home() / "pretrained_models" / "freudenberg2022.pt"

# ── Auflösungsparameter ────────────────────────────────────────────────────
KIEL_RES_M  = 0.075 if RESOLUTION == "7.5cm" else 0.20
MODEL_RES_M = 0.075 if RESOLUTION == "7.5cm" else 0.20

# ── Band-Reihenfolge im RGBI-GeoTIFF (1-basiert) ─────────────────────────
BAND_RED = 1
BAND_GRN = 2
BAND_BLU = 3
BAND_NIR = 4

# ── Inferenz ───────────────────────────────────────────────────────────────
PATCH_SIZE       = 682   #256
PATCH_STRIDE     = 341   #128
LOCAL_BATCH_SIZE = 16

# ── Post-Processing ────────────────────────────────────────────────────────
PP = dict(
    mask_exp           = 2,
    outline_multiplier = 5,
    outline_exp        = 1,
    dist_exp           = 0.5,
    sigma              = 1,
    binary_threshold   = 0.10,
    min_dist           = 10,
    label_threshold    = 0.10,
    area_min           = 3,
    simplify           = 0.3,
)

# ── Evaluation ─────────────────────────────────────────────────────────────
IOU_THRESHOLD = 0.5

print("Konfiguration geladen.")
print(f"Auflösung:   {RESOLUTION}")
print(f"Test-Gebiete: {TEST_AREAS}")
print(f"Modell-Pfad: {MODEL_PATH}")

Konfiguration geladen.
Auflösung:   20cm
Test-Gebiete: ['BotGarten', 'HoernNord']
Modell-Pfad: /home/leafline/pretrained_models/freudenberg2022.pt


## 1 — Imports

In [10]:
import json
import numpy as np
import torch
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import from_bounds
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import Polygon, shape
from tqdm import tqdm

from deeptrees.model.deeptrees_model import DeepTreesModel
from deeptrees.pretrained import freudenberg2022
from deeptrees.modules.utils import predict_on_tile
from deeptrees.modules.postprocessing import extract_polygons

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(torch.cuda.get_device_name(0))
print(torch.version.hip)

Device: cuda
AMD Radeon Graphics
7.2.53211


## 2 — Modell laden

### 2a — Gewichte herunterladen

`freudenberg2022()` lädt ein JIT-Traced `.pt`-File vom Helmholtz-Syncandshare-Server herunter (~170 MB). Der Download passiert nur einmal — danach wird das gecachte File genutzt.

In [11]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

if not MODEL_PATH.exists():
    print("Lade Modellgewichte herunter ...")
    freudenberg2022(str(MODEL_PATH))
    print(f"Gespeichert: {MODEL_PATH}")
else:
    print(f"Modell bereits vorhanden: {MODEL_PATH}")

Modell bereits vorhanden: /home/leafline/pretrained_models/freudenberg2022.pt


### 2b — Modell instantiieren und Gewichte übertragen

Das heruntergeladene File ist ein JIT-Script. Wir laden dessen State Dict in ein frisches `DeepTreesModel` (4 Eingangskanäle: R, G, B, IR). `apply_sigmoid=True` ist wichtig — das Modell gibt sonst rohe Logits aus, was `extract_polygons` nicht erwartet.

In [12]:
model = DeepTreesModel(
    in_channels   = 5,   # RGBI (4) + NDVI (1)
    architecture  = "Unet",
    backbone      = "resnet18",
    apply_sigmoid = True,
    num_backbones = 1,
)

jit_model = torch.jit.load(str(MODEL_PATH), map_location="cpu")
model.tcd_backbone.load_state_dict(jit_model.state_dict())

model.eval()
model = model.to(device)
print("Modell geladen und auf", device, "verschoben.")

Modell geladen und auf cuda verschoben.


## 3 — Hilfsfunktionen

### 3a — Kachel laden (RGBI + nDOM → 5 Kanäle)

Das freudenberg2022-Modell wurde auf 5 Kanälen trainiert: RGBI (4) + nDOM (1). Beide GeoTIFFs werden auf Modell-Auflösung (20 cm) skaliert und gestackt. Der nDOM-Kanal wird auf [0, 1] normiert.

In [13]:
def load_tile(area_name: str, src_res: float = KIEL_RES_M, tgt_res: float = MODEL_RES_M):
    """
    Lädt RGBI + NDVI für ein Gebiet, skaliert auf Zielauflösung.

    Returns:
        img (np.ndarray): Float32-Array (5, H, W) — RGBI [0,1] + NDVI [−1,1]
        transform: Rasterio Affin-Transform nach dem Skalieren
        crs: Koordinatenreferenzsystem
    """
    scale = src_res / tgt_res

    rgbi_path = TILES_DIR / f"{area_name}.tif"

    with rasterio.open(rgbi_path) as src:
        new_h = max(1, int(src.height * scale))
        new_w = max(1, int(src.width  * scale))

        rgbi = src.read(
            indexes    = [BAND_RED, BAND_GRN, BAND_BLU, BAND_NIR],
            out_shape  = (4, new_h, new_w),
            resampling = Resampling.average,
        ).astype(np.float32) / 255.0

        transform = src.transform * src.transform.scale(
            src.width / new_w, src.height / new_h
        )
        crs = src.crs

    nir  = rgbi[3]  # shape (H, W)
    red  = rgbi[0]
    ndvi = ((nir - red) / (nir + red + 1e-8))[np.newaxis]  # (1, H, W)

    img = np.concatenate([rgbi, ndvi], axis=0)  # (5, H, W)
    return img, transform, crs

### 3b — Inferenz auf einer Kachel

`predict_on_tile` schneidet die Kachel in überlappende Patches (256 × 256 px), schiebt sie als Batch durch das Modell und fügt die Ausgaben gewichtet wieder zusammen (Pyramidengewichtung, um Artefakte an Kanten zu minimieren).

Der Output hat 3 Kanäle:
- **Kanal 0 — Maske**: Wahrscheinlichkeit, dass ein Pixel zu einer Baumkrone gehört
- **Kanal 1 — Outline**: Wahrscheinlichkeit, dass ein Pixel eine Kronengrenze ist
- **Kanal 2 — Distanztransformation**: Normierte Distanz zum nächsten Nicht-Baum-Pixel

In [14]:
def run_inference(img: np.ndarray):
    """
    Führt DeepTrees-Inferenz auf einem (4, H, W) Float32-Array aus.

    Returns:
        mask (np.ndarray): (H, W) Baumkronen-Wahrscheinlichkeit
        outline (np.ndarray): (H, W) Outline-Wahrscheinlichkeit
        dist (np.ndarray): (H, W) Distanztransformation
    """
    tensor = torch.from_numpy(img).unsqueeze(0).to(device)  # (1, 4, H, W)

    with torch.no_grad():
        output = predict_on_tile(
            model,
            tensor,
            patch_size       = PATCH_SIZE,
            local_batch_size = LOCAL_BATCH_SIZE,
            stride           = PATCH_STRIDE,
        )  # → (1, 3, H, W)

    mask    = output[0, 0].cpu().numpy()
    outline = output[0, 1].cpu().numpy()
    dist    = output[0, 2].cpu().numpy()
    return mask, outline, dist

### 3c — GT-Annotationen laden

GT-Shapefiles liegen unter `TrainingAreas/test/{name}_GroundTruth.shp`.

In [15]:
def load_gt_polygons(area_name: str) -> list:
    """
    Lädt Ground-Truth-Polygone für ein Testgebiet.
    Erwartet: GT_DIR / {area_name}_GroundTruth.shp
    """
    gt_path = GT_DIR / f"{area_name}_GroundTruth.shp"
    gdf = gpd.read_file(gt_path)
    return list(gdf.geometry)

### 3d — Instanz-Evaluation (IoU-Matching)

Für jeden vorhergesagten Polygon suchen wir den am besten übereinstimmenden GT-Polygon (greedy, nach IoU sortiert). Liegt der beste IoU ≥ `IOU_THRESHOLD`, zählt es als **TP**. Ohne Match → **FP**. Nicht gematchte GT-Polygone → **FN**.

Daraus ergibt sich:
- **Precision** = TP / (TP + FP) — wie viele der gefundenen Bäume sind echt?
- **Recall** = TP / (TP + FN) — wie viele der echten Bäume wurden gefunden?
- **F1** = 2 · Precision · Recall / (Precision + Recall)

In [16]:
def iou_polygon(p1: Polygon, p2: Polygon) -> float:
    inter = p1.intersection(p2).area
    union = p1.union(p2).area
    return inter / union if union > 0 else 0.0


def match_polygons(pred: list, gt: list, threshold: float = IOU_THRESHOLD):
    """
    Greedy IoU-Matching: jeder GT-Polygon kann nur einmal gematcht werden.

    Returns:
        tp, fp, fn (int)
    """
    matched_gt = set()
    tp = 0

    for p in pred:
        best_iou  = 0.0
        best_idx  = -1
        for i, g in enumerate(gt):
            if i in matched_gt:
                continue
            iou = iou_polygon(p, g)
            if iou > best_iou:
                best_iou = iou
                best_idx = i
        if best_iou >= threshold:
            tp += 1
            matched_gt.add(best_idx)

    fp = len(pred) - tp
    fn = len(gt)   - len(matched_gt)
    return tp, fp, fn


def compute_metrics(tp: int, fp: int, fn: int):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

## 4 — Einzelne Kachel testen (Debugging / Visualisierung)

Bevor wir alle Kacheln durchlaufen, testen wir eine einzelne Kachel und schauen uns die Zwischenergebnisse an.

In [17]:
sample_area = TEST_AREAS[0]
print(f"Test-Gebiet : {sample_area}")

img, trafo, crs = load_tile(sample_area)
print(f"Bildgröße nach Skalierung: {img.shape}  (5, H, W)  — RGBI + nDOM")
print(f"Wertebereich: [{img.min():.3f}, {img.max():.3f}]")
print(f"CRS: {crs}")
print(f"tiles from: {TILES_DIR}")
print(f"model res: {MODEL_RES_M}")
print(f"patch_size: {PATCH_SIZE}")
print(f"patch_stride: {PATCH_STRIDE}")

Test-Gebiet : BotGarten
2026-07-01 10:23:57 - rasterio._env - INFO - GDAL signalled an error: err_no=4, msg='../Data/Kiel/TrainingAreas/DOP20/BotGarten.tif: Permission denied'


RasterioIOError: ../Data/Kiel/TrainingAreas/DOP20/BotGarten.tif: Permission denied

In [ ]:
# Inferenz auf der Test-Kachel
mask, outline, dist = run_inference(img)
print(f"Output-Shape: {mask.shape}")
print(f"Maske  min/max: {mask.min():.3f} / {mask.max():.3f}")
print(f"Outline min/max: {outline.min():.3f} / {outline.max():.3f}")
print(f"Dist   min/max: {dist.min():.3f} / {dist.max():.3f}")

In [ ]:
# Visualisierung der Modellausgaben
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# RGB aus den ersten 3 Bändern (R, G, B)
rgb = np.clip(img[:3].transpose(1, 2, 0), 0, 1)
axes[0].imshow(rgb)
axes[0].set_title("RGB (herunterskaliert)")
axes[0].axis("off")

axes[1].imshow(mask, cmap="Greens", vmin=0, vmax=1)
axes[1].set_title("Maske (Baumkrone)")
axes[1].axis("off")

axes[2].imshow(outline, cmap="Reds", vmin=0, vmax=1)
axes[2].set_title("Outline (Kronengrenzen)")
axes[2].axis("off")

axes[3].imshow(dist, cmap="Blues")
axes[3].set_title("Distanztransformation")
axes[3].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Post-Processing: Watershed → Baumkronen-Polygone
#
# extract_polygons kombiniert Maske, Outline und Distanztransformation:
#   1. Berechnet ein kombiniertes Wahrscheinlichkeits-Feature-Map
#   2. Wendet Gaussian Blur zur Glättung an
#   3. Findet lokale Maxima als Baum-Seeds (corner_peaks)
#   4. Führt Watershed-Segmentierung durch
#   5. Konvertiert gelabelte Regionen in Shapely-Polygone via rasterio.features.shapes
#
# Der `transform`-Parameter sorgt dafür, dass die Polygone in echten
# Koordinaten (CRS des GeoTIFF) ausgegeben werden.

polygons = extract_polygons(
    mask,
    outline,
    dist,
    transform = trafo,
    **PP
)

print(f"Gefundene Baumkronen: {len(polygons)}")

In [ ]:
# Polygone auf dem Bild anzeigen
from rasterio.transform import rowcol
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection

fig, ax = plt.subplots(1, 1, figsize=(12, 12))
ax.imshow(rgb)
ax.set_title(f"Erkannte Baumkronen: {len(polygons)}")
ax.axis("off")

# Koordinaten von Geo-Space → Pixel-Space umrechnen
inv_trafo = ~trafo
patches = []
for poly in polygons:
    xs, ys = poly.exterior.xy
    # (geo_x, geo_y) → (col, row) im herunterska lierten Bild
    cols, rows = inv_trafo * (np.array(xs), np.array(ys))
    pixel_coords = np.column_stack([cols, rows])
    patches.append(MplPolygon(pixel_coords, closed=True))

pc = PatchCollection(patches, facecolor="none", edgecolor="yellow", linewidth=0.8, alpha=0.9)
ax.add_collection(pc)
plt.tight_layout()
plt.show()

## 5 — Evaluation einer einzelnen Kachel

In [ ]:
gt_polygons = load_gt_polygons(sample_area)
print(f"GT-Bäume: {len(gt_polygons)}")
print(f"Vorhergesagte Bäume: {len(polygons)}")

In [ ]:
# IoU-Matching
tp, fp, fn = match_polygons(polygons, gt_polygons)
precision, recall, f1 = compute_metrics(tp, fp, fn)

print(f"TP (korrekt erkannte Bäume):      {tp}")
print(f"FP (falsch erkannte Nicht-Bäume): {fp}")
print(f"FN (übersehene Bäume):            {fn}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1:        {f1:.3f}")

In [ ]:
# Visualisierung: TP grün, FP rot, FN blau
matched_pred_idx = set()
matched_gt_idx   = set()

for pi, p in enumerate(polygons):
    best_iou, best_gi = 0.0, -1
    for gi, g in enumerate(gt_polygons):
        if gi in matched_gt_idx:
            continue
        iou = iou_polygon(p, g)
        if iou > best_iou:
            best_iou, best_gi = iou, gi
    if best_iou >= IOU_THRESHOLD:
        matched_pred_idx.add(pi)
        matched_gt_idx.add(best_gi)

fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(rgb)
ax.set_title("Grün=TP  Rot=FP  Blau=FN")
ax.axis("off")

for pi, poly in enumerate(polygons):
    color = "lime" if pi in matched_pred_idx else "red"
    xs, ys = poly.exterior.xy
    cols, rows = inv_trafo * (np.array(xs), np.array(ys))
    ax.fill(cols, rows, alpha=0.3, fc=color, ec=color, lw=1)

for gi, gt in enumerate(gt_polygons):
    if gi not in matched_gt_idx:
        xs, ys = gt.exterior.xy
        cols, rows = inv_trafo * (np.array(xs), np.array(ys))
        ax.fill(cols, rows, alpha=0.3, fc="blue", ec="blue", lw=1)

legend = [
    mpatches.Patch(color="lime",  label=f"TP ({tp})"),
    mpatches.Patch(color="red",   label=f"FP ({fp})"),
    mpatches.Patch(color="blue",  label=f"FN ({fn})"),
]
ax.legend(handles=legend, loc="upper right", fontsize=12)
plt.tight_layout()
plt.show()

## 6 — Evaluation über alle Kacheln

Hier wird die komplette Pipeline über alle gefundenen Kacheln ausgeführt und die Metriken aggregiert.

In [ ]:
import pandas as pd

EVAL_PLAN = [
    ("BotGarten",  "7.5cm"),
    ("BotGarten",  "20cm"),
    ("BotGarten",  "20cm-spring"),
    ("HoernNord",  "7.5cm"),
    ("HoernNord",  "20cm"),
    ("HoernNord",  "20cm-spring"),
]

results = []

#quickfix: resetting MODEL_RES_M, PATCH_SIZE and PATCH_STRIDE according to img resolution
for area, res in tqdm(EVAL_PLAN, desc="Gebiete"):
    try:
        if res == "7.5cm":
            t_dir   = BASE / "DOP7-5"
            src_res = 0.075
            MODEL_RES_M = 0.075
            PATCH_SIZE = 682 
            PATCH_STRIDE = 341           
        elif res == "20cm-spring":
            t_dir   = BASE / "DOP20-spring"
            src_res = 0.20
            MODEL_RES_M = 0.20
            PATCH_SIZE = 256
            PATCH_STRIDE = 128 
        else:
            t_dir   = BASE / "DOP20"
            src_res = 0.20
            MODEL_RES_M = 0.20
            PATCH_SIZE = 256
            PATCH_STRIDE = 128 

        rgbi_path = t_dir / f"{area}.tif"

        scale = src_res / MODEL_RES_M
        with rasterio.open(rgbi_path) as src:
            new_h = max(1, int(src.height * scale))
            new_w = max(1, int(src.width  * scale))
            rgbi = src.read(
                indexes=[BAND_RED, BAND_GRN, BAND_BLU, BAND_NIR],
                out_shape=(4, new_h, new_w),
                resampling=Resampling.average,
            ).astype(np.float32) / 255.0
            transform = src.transform * src.transform.scale(src.width / new_w, src.height / new_h)
            crs = src.crs

        nir  = rgbi[3]
        red  = rgbi[0]
        ndvi = ((nir - red) / (nir + red + 1e-8))[np.newaxis]  # (1, H, W)

        img = np.concatenate([rgbi, ndvi], axis=0)  # (5, H, W)
        mask, outline, dist = run_inference(img)

        pred_polys = extract_polygons(mask, outline, dist, transform=transform, **PP)
        gt_polys   = load_gt_polygons(area)

        tp, fp, fn = match_polygons(pred_polys, gt_polys)
        precision, recall, f1 = compute_metrics(tp, fp, fn)

        results.append({
            "gebiet":     area,
            "aufloesung": res,
            "pred":       len(pred_polys),
            "gt":         len(gt_polys),
            "tp":         tp,
            "fp":         fp,
            "fn":         fn,
            "precision":  precision,
            "recall":     recall,
            "f1":         f1,
        })

    except Exception as e:
        print(f"Fehler bei {area} ({res}): {e}")
        results.append({"gebiet": area, "aufloesung": res, "fehler": str(e)})

df = pd.DataFrame(results)
df

In [ ]:
import numpy as np

# Select only numeric columns
numeric_cols = df.select_dtypes(include=np.number).columns

# Mean and standard deviation
summary = pd.DataFrame({
    "mean": df[numeric_cols].mean(),
    "std": df[numeric_cols].std()
})

# Format as "mean ± std"
summary["mean ± std"] = summary.apply(
    lambda row: f"{row['mean']:.2f} ± {row['std']:.2f}",
    axis=1
)

summary = summary[["mean ± std"]]
print(summary)

In [ ]:
# Aggregierte Metriken über alle Kacheln
#
# Micro-Averaging: TP/FP/FN werden zuerst summiert, dann die Metriken berechnet.
# Das ist stabiler als der Durchschnitt der Per-Kachel-Metriken, weil es
# Kacheln mit wenigen Bäumen nicht übergewichtet.

total_tp = df["tp"].sum()
total_fp = df["fp"].sum()
total_fn = df["fn"].sum()

precision, recall, f1 = compute_metrics(total_tp, total_fp, total_fn)

print("=" * 40)
print("  BASELINE ERGEBNIS (Micro-Average)")
print("=" * 40)
print(f"  Kacheln:    {len(df)}")
print(f"  GT-Bäume:   {df['gt'].sum()}")
print(f"  Pred-Bäume: {df['pred'].sum()}")
print(f"  TP:         {total_tp}")
print(f"  FP:         {total_fp}")
print(f"  FN:         {total_fn}")
print(f"  Precision:  {precision:.3f}")
print(f"  Recall:     {recall:.3f}")
print(f"  F1:         {f1:.3f}")
print("=" * 40)

In [ ]:
output_path = Path.home() / "baseline_ergebnisse.csv"
df.to_csv(output_path, index=False)
print(f"Gespeichert: {output_path}")